In [73]:
import json
import time

import numpy as np
import pandas as pd

from bs4 import BeautifulSoup

from pydantic import BaseModel
from typing import List, Optional

from selenium import webdriver

from supabase import create_client, Client

In [74]:
driver = webdriver.Chrome()

In [75]:
whoscored_url = "https://www.whoscored.com/matches/1903417/live/england-premier-league-2025-2026-manchester-united-liverpool"
driver.get(whoscored_url)

In [76]:
soup = BeautifulSoup(driver.page_source, 'html.parser')

In [77]:
element = soup.select_one('script:-soup-contains("matchCentreData")')

In [78]:
matchdict = json.loads(element.text.split("matchCentreData: ")[1].split(',\n')[0])

In [79]:
matchdict.keys()

dict_keys(['playerIdNameDictionary', 'periodMinuteLimits', 'timeStamp', 'attendance', 'venueName', 'referee', 'weatherCode', 'elapsed', 'startTime', 'startDate', 'score', 'htScore', 'ftScore', 'etScore', 'pkScore', 'statusCode', 'periodCode', 'home', 'away', 'maxMinute', 'minuteExpanded', 'maxPeriod', 'expandedMinutes', 'expandedMaxMinute', 'periodEndMinutes', 'commonEvents', 'events', 'timeoutInSeconds'])

In [82]:
match_events = matchdict['events']

In [83]:
df = pd.DataFrame(match_events)

In [84]:
df.columns

Index(['id', 'eventId', 'minute', 'second', 'teamId', 'x', 'y',
       'expandedMinute', 'period', 'type', 'outcomeType', 'qualifiers',
       'satisfiedEventsTypes', 'isTouch', 'playerId', 'endX', 'endY',
       'relatedEventId', 'relatedPlayerId', 'blockedX', 'blockedY',
       'goalMouthZ', 'goalMouthY', 'isShot', 'isGoal', 'cardType'],
      dtype='str')

In [85]:
df.dropna(subset='playerId', inplace=True)

In [86]:
df = df.where(pd.notnull(df), None)

In [87]:
df = df.rename(
    {
        'eventId': 'event_id',
        'expandedMinute': 'expanded_minute',
        'outcomeType': 'outcome_type',
        'isTouch': 'is_touch',
        'playerId': 'player_id',
        'teamId': 'team_id',
        'endX': 'end_x',
        'endY': 'end_y',
        'blockedX': 'blocked_x',
        'blockedY': 'blocked_y',
        'goalMouthZ': 'goal_mouth_z',
        'goalMouthY': 'goal_mouth_y',
        'isShot': 'is_shot',
        'cardType': 'card_type',
        'isGoal': 'is_goal'
    },
    axis=1
)

In [88]:
df['period_display_name'] = df['period'].apply(lambda x: x['displayName'])
df['type_display_name'] = df['type'].apply(lambda x: x['displayName'])
df['outcome_type_display_name'] = df['outcome_type'].apply(lambda x: x['displayName'])

In [89]:
df.drop(columns=["period", "type", "outcome_type"], inplace=True)

In [90]:
if 'is_goal' not in df.columns:
    print('missing goals')
    df['is_goal'] = False

In [91]:
df = df[~(df['type_display_name'] == "OffsideGiven")]

In [92]:
df.columns

Index(['id', 'event_id', 'minute', 'second', 'team_id', 'x', 'y',
       'expanded_minute', 'qualifiers', 'satisfiedEventsTypes', 'is_touch',
       'player_id', 'end_x', 'end_y', 'relatedEventId', 'relatedPlayerId',
       'blocked_x', 'blocked_y', 'goal_mouth_z', 'goal_mouth_y', 'is_shot',
       'is_goal', 'card_type', 'period_display_name', 'type_display_name',
       'outcome_type_display_name'],
      dtype='str')

In [65]:
df.head()

,id,event_id,minute,second,team_id,x,y,expanded_minute,qualifiers,satisfiedEventsTypes,...,blocked_x,blocked_y,goal_mouth_z,goal_mouth_y,is_shot,is_goal,card_type,period_display_name,type_display_name,outcome_type_display_name
2,2.932943e+09,4,0,0.0,26,50.2,50.0,0,"[{'type': {'value': 212, 'displayName': 'Lengt...","[91, 117, 30, 35, 38, 215, 218]",...,NaN,NaN,NaN,NaN,None,None,None,FirstHalf,Pass,Successful
3,2.932943e+09,5,0,3.0,26,27.1,46.5,0,"[{'type': {'value': 141, 'displayName': 'PassE...","[91, 119, 117, 127, 205, 36, 37, 217, 218]",...,NaN,NaN,NaN,NaN,None,None,None,FirstHalf,Pass,Successful
4,2.932943e+09,6,0,8.0,26,74.3,71.2,0,"[{'type': {'value': 212, 'displayName': 'Lengt...","[91, 120, 139, 128, 36, 38, 217, 218]",...,NaN,NaN,NaN,NaN,None,None,None,FirstHalf,Pass,Unsuccessful
5,2.932943e+09,4,0,12.0,32,5.8,68.1,0,[],[93],...,NaN,NaN,NaN,NaN,None,None,None,FirstHalf,BallRecovery,Successful
6,2.932943e+09,5,0,29.0,32,14.3,49.9,0,"[{'type': {'value': 141, 'displayName': 'PassE...","[91, 117, 30, 36, 38, 215, 218]",...,NaN,NaN,NaN,NaN,None,None,None,FirstHalf,Pass,Successful


In [93]:
df.qualifiers.iloc[0]

[{'type': {'value': 212, 'displayName': 'Length'}, 'value': '24.9'},
 {'type': {'value': 56, 'displayName': 'Zone'}, 'value': 'Back'},
 {'type': {'value': 140, 'displayName': 'PassEndX'}, 'value': '26.5'},
 {'type': {'value': 141, 'displayName': 'PassEndY'}, 'value': '47.4'},
 {'type': {'value': 213, 'displayName': 'Angle'}, 'value': '3.21'}]

In [94]:
flat_qualifiers = []

for row in df['qualifiers']:
    row_data = {}
    if isinstance(row, list):
        for item in row:
            # displayName becomes the column name
            name = item.get('type', {}).get('displayName')
            # 'value' is the data; if it's missing, it's a 'tag' (so we mark as True)
            val = item.get('value', True)
            if name:
                row_data[name] = val
    flat_qualifiers.append(row_data)

# 2. Create the new DataFrame (Pandas handles the NaNs automatically)
qual_df = pd.DataFrame(flat_qualifiers, index=df.index)

# 3. Concatenate and convert numeric strings to actual floats/ints
df = pd.concat([df, qual_df], axis=1)

for col in qual_df.columns:
    try:
        # If it's a number, convert it.
        df[col] = pd.to_numeric(df[col])
    except (ValueError, TypeError):
        # If it's a string like 'Back' or 'Zone', leave it as is.
        continue

In [95]:
df.columns

Index(['id', 'event_id', 'minute', 'second', 'team_id', 'x', 'y',
       'expanded_minute', 'qualifiers', 'satisfiedEventsTypes',
       ...
       'LeadingToAttempt', 'BoxRight', 'LeadingToGoal', 'Yellow', 'LastMan',
       'IntentionalGoalAssist', 'SmallBoxRight', 'DirectFreekick', 'BoxLeft',
       'HighRight'],
      dtype='str', length=103)

In [70]:
df.shape[1]

103

In [71]:
df.columns

Index(['id', 'event_id', 'minute', 'second', 'team_id', 'x', 'y',
       'expanded_minute', 'qualifiers', 'satisfiedEventsTypes',
       ...
       'LeadingToAttempt', 'BoxRight', 'LeadingToGoal', 'Yellow', 'LastMan',
       'IntentionalGoalAssist', 'SmallBoxRight', 'DirectFreekick', 'BoxLeft',
       'HighRight'],
      dtype='str', length=103)

In [97]:
list(df)

['id',
 'event_id',
 'minute',
 'second',
 'team_id',
 'x',
 'y',
 'expanded_minute',
 'qualifiers',
 'satisfiedEventsTypes',
 'is_touch',
 'player_id',
 'end_x',
 'end_y',
 'relatedEventId',
 'relatedPlayerId',
 'blocked_x',
 'blocked_y',
 'goal_mouth_z',
 'goal_mouth_y',
 'is_shot',
 'is_goal',
 'card_type',
 'period_display_name',
 'type_display_name',
 'outcome_type_display_name',
 'Length',
 'Zone',
 'PassEndX',
 'PassEndY',
 'Angle',
 'Longball',
 'HeadPass',
 'LayOff',
 'Defensive',
 'OppositeRelatedEvent',
 'Offensive',
 'ThrowIn',
 'Chipped',
 'PlayerCaughtOffside',
 'Cross',
 'RightFoot',
 'FreekickTaken',
 'IndirectFreekickTaken',
 'LeftFoot',
 'GoalKick',
 'StandingSave',
 'IntentionalAssist',
 'ShotAssist',
 'KeyPass',
 'GoalMouthY',
 'BoxCentre',
 'RegularPlay',
 'FirstTouch',
 'BlockedY',
 'LowCentre',
 'Assisted',
 'RelatedEventId',
 'BlockedX',
 'GoalMouthZ',
 'Blocked',
 'OutfielderBlock',
 'MissLeft',
 'MissRight',
 'CornerTaken',
 'Head',
 'OutOfBoxCentre',
 'FromCo